# EEG Emotion Recognition - LOSO Pipeline

Овој notebook опфаќа:
- preprocessing на EEG сигнали
- екстракција на qEEG карактеристики
- SVM baseline
- EEGNet
- DeepConvNet
- LOSO евалуација

## 1. Импортирање на библиотеки

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io

from scipy.io import loadmat
from scipy.signal import butter, filtfilt, welch, resample

from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## 2. Helper functions

In [2]:
def bandpass_filter(data, lowcut=1, highcut=45, fs=256, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype='band')
    filtered = filtfilt(b, a, data, axis=-1)

    return filtered

In [3]:
def segment_signal(data, window_size=1024, step_size=512):
    """
    data shape: (channels, samples)
    returns shape: (num_segments, channels, window_size)
    """
    segments = []
    n_samples = data.shape[1]

    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)

    return np.array(segments)

In [4]:
def zscore_with_stats(signal, mean, std):
    return (signal - mean) / (std + 1e-8)

def compute_subject_stats(trials):
    """
    trials: list of arrays with shape [channels, time]
    returns mean/std per channel computed across all trials of one subject
    """
    all_data = np.concatenate(trials, axis=1)
    mean = np.mean(all_data, axis=1, keepdims=True)
    std = np.std(all_data, axis=1, keepdims=True)
    return mean, std

In [6]:
def resample_signal(signal, orig_fs, target_fs):
    num_samples = int(signal.shape[1] * target_fs / orig_fs)
    return resample(signal, num_samples, axis=1)

In [8]:
def extract_bandpower(segment, fs=256):
    bands = {
        'delta': (1, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 45)
    }

    features = []

    for ch in segment:
        freqs, psd = welch(ch, fs=fs)

        for band in bands.values():
            low, high = band
            idx = (freqs >= low) & (freqs <= high)
            band_power = np.mean(psd[idx])
            features.append(band_power)

    return np.array(features)

In [10]:
def extract_features_from_segments(segments, fs=256):
    feature_matrix = []

    for segment in segments:
        features = extract_bandpower(segment, fs=fs)
        feature_matrix.append(features)

    return np.array(feature_matrix)

In [13]:
def create_segment_labels(num_segments, trial_label):
    """
    num_segments: број на сегменти од едно recording/trial
    trial_label: 0 или 1
    returns shape: (num_segments,)
    """
    return np.full(num_segments, trial_label)

In [14]:
def create_subject_ids(num_segments, subject_id):
    """
    num_segments: број на сегменти
    subject_id: идентификатор на subject
    returns shape: (num_segments,)
    """
    return np.full(num_segments, subject_id)

In [18]:
def prepare_eegnet_data(segments):
    """
    segments shape: (num_segments, channels, samples)
    returns: (num_segments, 1, channels, samples)
    """
    return segments[:, np.newaxis, :, :]

In [24]:
def train_eegnet(model, X_train, y_train, epochs=10, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    return model

In [25]:
def evaluate_eegnet(model, X_test, y_test):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    with torch.no_grad():
        outputs = model(X_test)
        predictions = torch.argmax(outputs, dim=1).cpu().numpy()

    acc = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, zero_division=0)

    return acc, f1, predictions

In [28]:
def run_loso_svm(X, y, groups):
    logo = LeaveOneGroupOut()

    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    macro_f1_scores = []
    conf_matrices = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
        print(f"\nFold {fold}/{len(np.unique(groups))}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        print("Train class counts:", np.bincount(y_train))
        print("Test class counts:", np.bincount(y_test))

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = SVC(
            kernel='rbf',
            C=1.0,
            gamma='scale',
            class_weight='balanced'
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        cm = confusion_matrix(y_test, y_pred)

        print(f"  Fold Accuracy: {acc:.4f}")
        print(f"  Fold Precision: {prec:.4f}")
        print(f"  Fold Recall: {rec:.4f}")
        print(f"  Fold F1-score: {f1:.4f}")
        print(f"  Fold Macro F1: {f1_macro:.4f}")
        print(f"  Confusion Matrix:\n{cm}")

        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
        macro_f1_scores.append(f1_macro)
        conf_matrices.append(cm)

    return {
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std": np.std(accuracies),
        "precision_mean": np.mean(precisions),
        "precision_std": np.std(precisions),
        "recall_mean": np.mean(recalls),
        "recall_std": np.std(recalls),
        "f1_mean": np.mean(f1_scores),
        "f1_std": np.std(f1_scores),
        "macro_f1_mean": np.mean(macro_f1_scores),
        "macro_f1_std": np.std(macro_f1_scores),
        "confusion_matrices": conf_matrices
    }

In [30]:
def run_loso_eegnet(X, y, groups, num_channels=14, num_samples=1024, epochs=5, lr=0.001, batch_size=128):
    logo = LeaveOneGroupOut()

    accuracies = []
    f1_scores = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
        print(f"\nFold {fold}/{len(np.unique(groups))}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        X_test_t = torch.tensor(X_test, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.long)
        y_test_t = torch.tensor(y_test, dtype=torch.long)

        train_loader = DataLoader(
            TensorDataset(X_train_t, y_train_t),
            batch_size=batch_size,
            shuffle=True
        )

        test_loader = DataLoader(
            TensorDataset(X_test_t, y_test_t),
            batch_size=batch_size,
            shuffle=False
        )

        model = EEGNet(num_channels=num_channels, num_samples=num_samples, num_classes=2).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        model.train()
        for epoch in range(epochs):
            epoch_loss = 0.0

            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)

                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()

            print(f"  Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / len(train_loader):.4f}")

        model.eval()
        all_preds = []
        all_true = []

        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)

                outputs = model(batch_X)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                all_preds.extend(preds)
                all_true.extend(batch_y.numpy())

        acc = accuracy_score(all_true, all_preds)
        f1 = f1_score(all_true, all_preds, zero_division=0)

        accuracies.append(acc)
        f1_scores.append(f1)

        print(f"  Fold Accuracy: {acc:.4f}")
        print(f"  Fold F1-score: {f1:.4f}")

    return np.mean(accuracies), np.mean(f1_scores)

In [31]:
def resample_eeg(data, orig_fs=128, target_fs=256):
    """
    data shape: (channels, samples)
    """
    if orig_fs == target_fs:
        return data

    num_samples = int(data.shape[1] * target_fs / orig_fs)
    return resample(data, num_samples, axis=1)

## 3. Пример податоци за тестирање на pipeline

In [41]:
fake_eeg = np.random.randn(14, 5000)
segments = segment_signal(fake_eeg, window_size=1024, step_size=512)

print("Original shape:", fake_eeg.shape)
print("Segmented shape:", segments.shape)

Original shape: (14, 5000)
Segmented shape: (8, 14, 1024)


In [42]:
X_fake = extract_features_from_segments(segments, fs=256)
y_fake = create_segment_labels(num_segments=segments.shape[0], trial_label=1)
groups_fake = create_subject_ids(num_segments=segments.shape[0], subject_id=3)

print("Feature matrix shape:", X_fake.shape)
print("Label vector shape:", y_fake.shape)
print("Groups shape:", groups_fake.shape)

Feature matrix shape: (8, 70)
Label vector shape: (8,)
Groups shape: (8,)


## 4. SVM baseline

In [43]:
# Fake data for 4 subjects
X_all = []
y_all = []
groups_all = []

for subject_id in range(4):
    fake_eeg = np.random.randn(14, 5000)
    segments = segment_signal(fake_eeg, window_size=1024, step_size=512)
    X_subject = extract_features_from_segments(segments, fs=256)

    num_segments = X_subject.shape[0]
    half = num_segments // 2
    y_subject = np.array([0] * half + [1] * (num_segments - half))
    g_subject = create_subject_ids(num_segments=num_segments, subject_id=subject_id)

    X_all.append(X_subject)
    y_all.append(y_subject)
    groups_all.append(g_subject)

X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)
groups_all = np.concatenate(groups_all)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)
print("groups_all shape:", groups_all.shape)
print("Unique labels:", np.unique(y_all))
print("Unique groups:", np.unique(groups_all))

X_all shape: (32, 70)
y_all shape: (32,)
groups_all shape: (32,)
Unique labels: [0 1]
Unique groups: [0 1 2 3]


## 5. EEGNet

In [39]:
class EEGNet(nn.Module):
    def __init__(self, num_channels=14, num_samples=1024, num_classes=2,
                 F1=8, D=2, F2=16, kernel_length=64, dropout=0.5):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),

            nn.Conv2d(F1, F1 * D, (num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout)
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_samples)
            out = self.block1(dummy)
            out = self.block2(out)
            flattened_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Linear(flattened_dim, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [45]:
X_eegnet = prepare_eegnet_data(segments)
print("EEGNet input shape:", X_eegnet.shape)

EEGNet input shape: (8, 1, 14, 1024)


In [33]:
model = EEGNet()

x = torch.randn(8, 1, 14, 1024)
output = model(x)

print("Output shape:", output.shape)

Output shape: torch.Size([8, 2])


In [34]:
y_eegnet = np.array([0, 0, 0, 0, 1, 1, 1, 1])

model = EEGNet(num_channels=14, num_samples=1024, num_classes=2)
trained_model = train_eegnet(model, X_eegnet, y_eegnet, epochs=5, lr=0.001)

NameError: name 'X_eegnet' is not defined

In [48]:
acc, f1, preds = evaluate_eegnet(trained_model, X_eegnet, y_eegnet)

print("EEGNet Accuracy:", acc)
print("EEGNet F1-score:", f1)
print("Predictions:", preds)

EEGNet Accuracy: 0.5
EEGNet F1-score: 0.0
Predictions: [0 0 0 0 0 0 0 0]


In [44]:
X_eeg_all = []
y_eeg_all = []
groups_eeg_all = []

for subject_id in range(4):
    fake_eeg = np.random.randn(14, 5000)
    segments = segment_signal(fake_eeg, window_size=1024, step_size=512)

    X_subject = prepare_eegnet_data(segments)

    num_segments = X_subject.shape[0]
    half = num_segments // 2
    y_subject = np.array([0] * half + [1] * (num_segments - half))
    g_subject = create_subject_ids(num_segments=num_segments, subject_id=subject_id)

    X_eeg_all.append(X_subject)
    y_eeg_all.append(y_subject)
    groups_eeg_all.append(g_subject)

X_eeg_all = np.vstack(X_eeg_all)
y_eeg_all = np.concatenate(y_eeg_all)
groups_eeg_all = np.concatenate(groups_eeg_all)

print("X_eeg_all shape:", X_eeg_all.shape)
print("y_eeg_all shape:", y_eeg_all.shape)
print("groups_eeg_all shape:", groups_eeg_all.shape)

X_eeg_all shape: (32, 1, 14, 1024)
y_eeg_all shape: (32,)
groups_eeg_all shape: (32,)


In [45]:
mean_acc_eegnet, mean_f1_eegnet = run_loso_eegnet(
    X_eeg_all,
    y_eeg_all,
    groups_eeg_all,
    num_channels=14,
    num_samples=1024,
    epochs=5,
    lr=0.001
)

print("Mean LOSO EEGNet Accuracy:", mean_acc_eegnet)
print("Mean LOSO EEGNet F1-score:", mean_f1_eegnet)

Using device: cuda

Fold 1/4
  Epoch 1/5, Loss: 0.7500
  Epoch 2/5, Loss: 0.6727
  Epoch 3/5, Loss: 0.6749
  Epoch 4/5, Loss: 0.7364
  Epoch 5/5, Loss: 0.6787
  Fold Accuracy: 0.5000
  Fold F1-score: 0.5000

Fold 2/4
  Epoch 1/5, Loss: 0.7342
  Epoch 2/5, Loss: 0.6524
  Epoch 3/5, Loss: 0.7629
  Epoch 4/5, Loss: 0.6856
  Epoch 5/5, Loss: 0.6833
  Fold Accuracy: 0.6250
  Fold F1-score: 0.5714

Fold 3/4
  Epoch 1/5, Loss: 0.7154
  Epoch 2/5, Loss: 0.6887
  Epoch 3/5, Loss: 0.6575
  Epoch 4/5, Loss: 0.6826
  Epoch 5/5, Loss: 0.6604
  Fold Accuracy: 0.5000
  Fold F1-score: 0.0000

Fold 4/4
  Epoch 1/5, Loss: 0.7042
  Epoch 2/5, Loss: 0.6756
  Epoch 3/5, Loss: 0.6760
  Epoch 4/5, Loss: 0.6318
  Epoch 5/5, Loss: 0.5996
  Fold Accuracy: 0.5000
  Fold F1-score: 0.3333
Mean LOSO EEGNet Accuracy: 0.53125
Mean LOSO EEGNet F1-score: 0.35119047619047616


## 6. DeepConvNet

In [46]:
class DeepConvNet(nn.Module):
    def __init__(self, num_channels=14, num_samples=1024, num_classes=2):
        super(DeepConvNet, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 25, kernel_size=(1, 5)),
            nn.Conv2d(25, 25, kernel_size=(num_channels, 1)),
            nn.BatchNorm2d(25),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout(0.5)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(25, 50, kernel_size=(1, 5)),
            nn.BatchNorm2d(50),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout(0.5)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(50, 100, kernel_size=(1, 5)),
            nn.BatchNorm2d(100),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout(0.5)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(100, 200, kernel_size=(1, 5)),
            nn.BatchNorm2d(200),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout(0.5)
        )

        dummy = torch.randn(1, 1, num_channels, num_samples)
        dummy = self._forward_features(dummy)
        self.flattened_size = dummy.view(1, -1).shape[1]

        self.fc = nn.Linear(self.flattened_size, num_classes)

    def _forward_features(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [47]:
model = DeepConvNet()

x = torch.randn(8, 1, 14, 1024)
output = model(x)

print("DeepConvNet output shape:", output.shape)

DeepConvNet output shape: torch.Size([8, 2])


In [53]:
trained_model = train_eegnet(model, X_eegnet, y_eegnet, epochs=5, lr=0.001)

Epoch 1/5, Loss: 0.8401
Epoch 2/5, Loss: 0.9161
Epoch 3/5, Loss: 0.9933
Epoch 4/5, Loss: 1.0950
Epoch 5/5, Loss: 0.8996


In [48]:
acc, f1, preds = evaluate_eegnet(trained_model, X_eegnet, y_eegnet)

print("DeepConvNet Accuracy:", acc)
print("DeepConvNet F1:", f1)
print("Predictions:", preds)

NameError: name 'trained_model' is not defined

## 7. DEAP dataset

### 8.1 Преземање на DEAP

In [55]:
import pickle

In [49]:
import kagglehub

path_deap = kagglehub.dataset_download("manh123df/deap-dataset")
print("DEAP path:", path_deap)

DEAP path: /kaggle/input/datasets/manh123df/deap-dataset


### 8.2 Define exact directory

In [50]:
import os

deap_dir = os.path.join(path_deap, "deap-dataset", "data_preprocessed_python")
print("DEAP dir:", deap_dir)
print("Exists:", os.path.exists(deap_dir))
print("Example files:", os.listdir(deap_dir)[:5])

DEAP dir: /kaggle/input/datasets/manh123df/deap-dataset/deap-dataset/data_preprocessed_python
Exists: True
Example files: ['s20.dat', 's17.dat', 's31.dat', 's14.dat', 's32.dat']


### 8.3 List subject files

In [51]:
deap_files = sorted([
    os.path.join(deap_dir, f)
    for f in os.listdir(deap_dir)
    if f.endswith(".dat")
])

print("Number of DEAP subject files:", len(deap_files))
print(deap_files[:3])

Number of DEAP subject files: 32
['/kaggle/input/datasets/manh123df/deap-dataset/deap-dataset/data_preprocessed_python/s01.dat', '/kaggle/input/datasets/manh123df/deap-dataset/deap-dataset/data_preprocessed_python/s02.dat', '/kaggle/input/datasets/manh123df/deap-dataset/deap-dataset/data_preprocessed_python/s03.dat']


### 8.4 Extraction of EEG trials and valence labels

In [52]:
def load_deap_subject(subject_path, label_threshold=5.0):
    """
    subject_path: path to one DEAP .dat file
    returns:
        X_subject: (num_segments, 32, 1024)
        y_subject: (num_segments,)
    """
    with open(subject_path, "rb") as f:
        subject = pickle.load(f, encoding="latin1")

    data = subject["data"]
    labels = subject["labels"]

    processed_trials = []
    trial_labels = []

    for i in range(data.shape[0]):
        trial = data[i, :32, :]

        trial = trial[:, 384:]

        trial = bandpass_filter(trial, lowcut=1, highcut=45, fs=128, order=4)
        trial = resample_signal(trial, orig_fs=128, target_fs=256)

        processed_trials.append(trial)

        valence = labels[i, 0]
        label = 1 if valence > label_threshold else 0
        trial_labels.append(label)

    mean, std = compute_subject_stats(processed_trials)

    X_subject = []
    y_subject = []

    for trial, label in zip(processed_trials, trial_labels):
        trial_norm = zscore_with_stats(trial, mean, std)
        segments = segment_signal(trial_norm, window_size=1024, step_size=512)

        X_subject.append(segments)
        y_subject.extend([label] * len(segments))

    X_subject = np.vstack(X_subject).astype(np.float32)
    y_subject = np.array(y_subject, dtype=np.int64)

    return X_subject, y_subject

In [56]:
X_segments_sub1, y_labels_sub1 = load_deap_subject(deap_files[0], label_threshold=5)

print("DEAP subject 1 segments shape:", X_segments_sub1.shape)
print("DEAP subject 1 labels shape:", y_labels_sub1.shape)
print("Unique labels:", np.unique(y_labels_sub1))
print("Class counts:", np.bincount(y_labels_sub1))

DEAP subject 1 segments shape: (1160, 32, 1024)
DEAP subject 1 labels shape: (1160,)
Unique labels: [0 1]
Class counts: [986 174]


### 8.5 Подготовка на сите subjects за DEAP

In [57]:
X_deap = []
y_deap = []
groups_deap = []

num_subjects = len(deap_files)

for subject_id, subject_path in enumerate(deap_files):
    X_sub, y_sub = load_deap_subject(subject_path, label_threshold=5)
    g_sub = create_subject_ids(num_segments=len(y_sub), subject_id=subject_id)

    X_deap.append(X_sub)
    y_deap.append(y_sub)
    groups_deap.append(g_sub)

    print(f"Subject {subject_id+1}: segments={X_sub.shape[0]}, class counts={np.bincount(y_sub)}")

X_deap = np.vstack(X_deap)
y_deap = np.concatenate(y_deap)
groups_deap = np.concatenate(groups_deap)

print("\nFinal DEAP shapes:")
print("X_deap:", X_deap.shape)
print("y_deap:", y_deap.shape)
print("groups_deap:", groups_deap.shape)
print("Unique labels:", np.unique(y_deap))
print("Unique groups:", np.unique(groups_deap))

Subject 1: segments=1160, class counts=[986 174]
Subject 2: segments=1160, class counts=[986 174]
Subject 3: segments=1160, class counts=[551 609]
Subject 4: segments=1160, class counts=[1044  116]
Subject 5: segments=1160, class counts=[841 319]
Subject 6: segments=1160, class counts=[667 493]
Subject 7: segments=1160, class counts=[870 290]
Subject 8: segments=1160, class counts=[928 232]
Subject 9: segments=1160, class counts=[1015  145]
Subject 10: segments=1160, class counts=[899 261]
Subject 11: segments=1160, class counts=[725 435]
Subject 12: segments=1160, class counts=[1102   58]
Subject 13: segments=1160, class counts=[1102   58]
Subject 14: segments=1160, class counts=[928 232]
Subject 15: segments=1160, class counts=[812 348]
Subject 16: segments=1160, class counts=[986 174]
Subject 17: segments=1160, class counts=[899 261]
Subject 18: segments=1160, class counts=[957 203]
Subject 19: segments=1160, class counts=[1015  145]
Subject 20: segments=1160, class counts=[986 174]

### 8.6 LOSO евалуација на DEAP со SVM

In [ ]:
X_deap_features = extract_features_from_segments(X_deap, fs=256)
print("X_deap_features shape:", X_deap_features.shape)

In [ ]:
np.save("/content/X_deap_features.npy", X_deap_features)
np.save("/content/y_deap.npy", y_deap)
np.save("/content/groups_deap.npy", groups_deap)

In [ ]:
X_deap_features = np.load("/content/X_deap_features.npy")
y_deap = np.load("/content/y_deap.npy")
groups_deap = np.load("/content/groups_deap.npy")

In [ ]:
svm_results_deap = run_loso_svm(
    X_deap_features,
    y_deap,
    groups_deap
)

print("\n===== DEAP LOSO SVM RESULTS =====")
print(f"Accuracy:  {svm_results_deap['accuracy_mean']:.4f} ± {svm_results_deap['accuracy_std']:.4f}")
print(f"Precision: {svm_results_deap['precision_mean']:.4f} ± {svm_results_deap['precision_std']:.4f}")
print(f"Recall:    {svm_results_deap['recall_mean']:.4f} ± {svm_results_deap['recall_std']:.4f}")
print(f"F1-score:  {svm_results_deap['f1_mean']:.4f} ± {svm_results_deap['f1_std']:.4f}")
print(f"Macro F1:  {svm_results_deap['macro_f1_mean']:.4f} ± {svm_results_deap['macro_f1_std']:.4f}")

In [ ]:
results_deap_svm = {
    "accuracy_mean": svm_results_deap["accuracy_mean"],
    "accuracy_std": svm_results_deap["accuracy_std"],
    "precision_mean": svm_results_deap["precision_mean"],
    "precision_std": svm_results_deap["precision_std"],
    "recall_mean": svm_results_deap["recall_mean"],
    "recall_std": svm_results_deap["recall_std"],
    "f1_mean": svm_results_deap["f1_mean"],
    "f1_std": svm_results_deap["f1_std"],
    "macro_f1_mean": svm_results_deap["macro_f1_mean"],
    "macro_f1_std": svm_results_deap["macro_f1_std"],
}
np.save("deap_svm_results.npy", results_deap_svm)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

save_dir = "/content/drive/MyDrive/diplomska_eeg"
os.makedirs(save_dir, exist_ok=True)
print(save_dir)

## 9. EEGNet LOSO on DEAP

In [72]:
save_dir_raw_deap = "/kaggle/working/deap_eeg_raw"
os.makedirs(save_dir_raw_deap, exist_ok=True)

num_subjects = len(deap_files)

for subject_id, subject_path in enumerate(deap_files):
    X_sub, y_sub = load_deap_subject(subject_path, label_threshold=5)

    X_sub = X_sub.astype(np.float32)
    y_sub = y_sub.astype(np.int8)

    np.save(os.path.join(save_dir_raw_deap, f"X_subject_{subject_id:02d}.npy"), X_sub)
    np.save(os.path.join(save_dir_raw_deap, f"y_subject_{subject_id:02d}.npy"), y_sub)

    print(f"Saved subject {subject_id+1}/{num_subjects} -> X: {X_sub.shape}, y: {y_sub.shape}, class counts: {np.bincount(y_sub)}")

    del X_sub, y_sub
    gc.collect()

Saved subject 1/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [986 174]
Saved subject 2/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [986 174]
Saved subject 3/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [551 609]
Saved subject 4/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [1044  116]
Saved subject 5/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [841 319]
Saved subject 6/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [667 493]
Saved subject 7/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [870 290]
Saved subject 8/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [928 232]
Saved subject 9/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [1015  145]
Saved subject 10/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [899 261]
Saved subject 11/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [725 435]
Saved subject 12/32 -> X: (1160, 32, 1024), y: (1160,), class counts: [1102   58]
Saved subject 13/32 -> X: (1160, 32, 1024),

In [73]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [74]:
import os

print(os.path.exists("/kaggle/working/deap_eeg_raw"))
print(os.listdir("/kaggle/working/deap_eeg_raw")[:10])

True
['X_subject_04.npy', 'y_subject_17.npy', 'X_subject_28.npy', 'y_subject_15.npy', 'y_subject_27.npy', 'X_subject_13.npy', 'y_subject_07.npy', 'y_subject_16.npy', 'X_subject_16.npy', 'X_subject_03.npy']


In [ ]:
def run_loso_eegnet_from_disk(save_dir, num_subjects=23, num_channels=14, num_samples=1024,
                              epochs=3, lr=0.001, batch_size=32):
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    macro_f1_scores = []
    conf_matrices = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    for test_subject in range(num_subjects):
        print(f"\nFold {test_subject+1}/{num_subjects}")

        train_subjects = [i for i in range(num_subjects) if i != test_subject]

        # test data
        X_test = np.load(os.path.join(save_dir, f"X_subject_{test_subject:02d}.npy"))
        y_test = np.load(os.path.join(save_dir, f"y_subject_{test_subject:02d}.npy"))

        print("Test class counts:", np.bincount(y_test))

        # class weights from train labels only
        y_train_all = []
        for subject_id in train_subjects:
            y_sub = np.load(os.path.join(save_dir, f"y_subject_{subject_id:02d}.npy"))
            y_train_all.append(y_sub)

        y_train_all = np.concatenate(y_train_all)
        print("Train class counts:", np.bincount(y_train_all))

        classes = np.array([0, 1])
        weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_all)
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        model = EEGNet(
            num_channels=num_channels,
            num_samples=num_samples,
            num_classes=2,
            F1=8,
            D=2,
            F2=16,
            kernel_length=64,
            dropout=0.5
        ).to(device)

        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            batch_count = 0

            for subject_id in train_subjects:
                X_sub = np.load(os.path.join(save_dir, f"X_subject_{subject_id:02d}.npy"), mmap_mode='r')
                y_sub = np.load(os.path.join(save_dir, f"y_subject_{subject_id:02d}.npy"), mmap_mode='r')

                X_sub = np.array(X_sub, dtype=np.float32, copy=True)
                y_sub = np.array(y_sub, dtype=np.int64, copy=True)

                X_sub = X_sub[:, np.newaxis, :, :]
                train_ds = TensorDataset(torch.from_numpy(X_sub), torch.from_numpy(y_sub))
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

                for xb, yb in train_loader:
                    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

                    optimizer.zero_grad()
                    outputs = model(xb)
                    loss = criterion(outputs, yb)
                    loss.backward()
                    optimizer.step()

                    epoch_loss += loss.item()
                    batch_count += 1

                del X_sub, y_sub, train_ds, train_loader
                gc.collect()

            print(f"  Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / max(batch_count, 1):.4f}")

        model.eval()
        X_test = X_test[:, np.newaxis, :, :].astype(np.float32)
        test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test.astype(np.int64)))
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

        all_preds = []
        all_true = []

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device, non_blocking=True)
                outputs = model(xb)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                all_preds.extend(preds)
                all_true.extend(yb.numpy())

        all_preds = np.array(all_preds)
        all_true = np.array(all_true)

        acc = accuracy_score(all_true, all_preds)
        prec = precision_score(all_true, all_preds, zero_division=0)
        rec = recall_score(all_true, all_preds, zero_division=0)
        f1 = f1_score(all_true, all_preds, zero_division=0)
        f1_macro = f1_score(all_true, all_preds, average='macro', zero_division=0)
        cm = confusion_matrix(all_true, all_preds)

        print(f"  Fold Accuracy: {acc:.4f}")
        print(f"  Fold Precision: {prec:.4f}")
        print(f"  Fold Recall: {rec:.4f}")
        print(f"  Fold F1-score: {f1:.4f}")
        print(f"  Fold Macro F1: {f1_macro:.4f}")
        print(f"  Confusion Matrix:\n{cm}")

        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
        macro_f1_scores.append(f1_macro)
        conf_matrices.append(cm)

        del X_test, y_test, test_ds, test_loader, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std": np.std(accuracies),
        "precision_mean": np.mean(precisions),
        "precision_std": np.std(precisions),
        "recall_mean": np.mean(recalls),
        "recall_std": np.std(recalls),
        "f1_mean": np.mean(f1_scores),
        "f1_std": np.std(f1_scores),
        "macro_f1_mean": np.mean(macro_f1_scores),
        "macro_f1_std": np.std(macro_f1_scores),
        "confusion_matrices": conf_matrices
    }

In [ ]:
eegnet_results_deap = run_loso_eegnet_from_disk(
    save_dir=save_dir_raw_deap,
    num_subjects=32,
    num_channels=32,
    num_samples=1024,
    epochs=3,
    batch_size=64,
    lr=0.001,
)

print("\n===== DEAP LOSO EEGNet RESULTS =====")
print(f"Accuracy:  {eegnet_results_deap['accuracy_mean']:.4f} ± {eegnet_results_deap['accuracy_std']:.4f}")
print(f"Precision: {eegnet_results_deap['precision_mean']:.4f} ± {eegnet_results_deap['precision_std']:.4f}")
print(f"Recall:    {eegnet_results_deap['recall_mean']:.4f} ± {eegnet_results_deap['recall_std']:.4f}")
print(f"F1-score:  {eegnet_results_deap['f1_mean']:.4f} ± {eegnet_results_deap['f1_std']:.4f}")
print(f"Macro F1:  {eegnet_results_deap['macro_f1_mean']:.4f} ± {eegnet_results_deap['macro_f1_std']:.4f}")

In [ ]:
np.save("deap_eegnet_results.npy", eegnet_results_deap)
print("DEAP EEGNet results saved.")

## 10. DeepConvNet LOSO on DEAP

In [61]:
save_dir_raw = save_dir_raw_deap
print(save_dir_raw)

/content/deap_eeg_raw


In [ ]:
save_dir_raw = "/content/diplomska_eeg_raw"
print(save_dir_raw)

In [62]:
save_dir_raw = "/kaggle/working/deap_eeg_raw"

In [65]:
class DeepConvNet(nn.Module):
    def __init__(self, num_channels=14, num_samples=1024, num_classes=2, dropout=0.5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 25, (1, 10), stride=1, padding=0, bias=False),
            nn.Conv2d(25, 25, (num_channels, 1), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(25),
            nn.ELU(),
            nn.MaxPool2d((1, 3)),
            nn.Dropout(dropout),

            nn.Conv2d(25, 50, (1, 10), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(50),
            nn.ELU(),
            nn.MaxPool2d((1, 3)),
            nn.Dropout(dropout),

            nn.Conv2d(50, 100, (1, 10), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(100),
            nn.ELU(),
            nn.MaxPool2d((1, 3)),
            nn.Dropout(dropout),

            nn.Conv2d(100, 200, (1, 10), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(200),
            nn.ELU(),
            nn.MaxPool2d((1, 3)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_samples)
            out = self.features(dummy)
            flattened_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Linear(flattened_dim, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [76]:
def run_loso_deepconvnet_from_disk(save_dir, num_subjects=23, num_channels=14, num_samples=1024,
                                   epochs=3, lr=0.001, batch_size=64):
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    macro_f1_scores = []
    conf_matrices = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    for test_subject in range(num_subjects):
        print(f"\nFold {test_subject+1}/{num_subjects}")

        train_subjects = [i for i in range(num_subjects) if i != test_subject]

        X_test = np.load(os.path.join(save_dir, f"X_subject_{test_subject:02d}.npy"))
        y_test = np.load(os.path.join(save_dir, f"y_subject_{test_subject:02d}.npy"))

        print("Test class counts:", np.bincount(y_test))

        y_train_all = []
        for subject_id in train_subjects:
            y_sub = np.load(os.path.join(save_dir, f"y_subject_{subject_id:02d}.npy"))
            y_train_all.append(y_sub)

        y_train_all = np.concatenate(y_train_all)
        print("Train class counts:", np.bincount(y_train_all))

        classes = np.array([0, 1])
        weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_all)
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        model = DeepConvNet(
            num_channels=num_channels,
            num_samples=num_samples,
            num_classes=2,
            dropout=0.5
        ).to(device)

        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            batch_count = 0

            for subject_id in train_subjects:
                X_sub = np.load(os.path.join(save_dir, f"X_subject_{subject_id:02d}.npy"), mmap_mode='r')
                y_sub = np.load(os.path.join(save_dir, f"y_subject_{subject_id:02d}.npy"), mmap_mode='r')

                X_sub = np.array(X_sub, dtype=np.float32, copy=True)
                y_sub = np.array(y_sub, dtype=np.int64, copy=True)

                X_sub = X_sub[:, np.newaxis, :, :]
                train_ds = TensorDataset(torch.from_numpy(X_sub), torch.from_numpy(y_sub))
                train_loader = DataLoader(
                    train_ds,
                    batch_size=batch_size,
                    shuffle=True,
                    num_workers=0,
                    pin_memory=True
                )

                for xb, yb in train_loader:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)

                    optimizer.zero_grad()
                    outputs = model(xb)
                    loss = criterion(outputs, yb)
                    loss.backward()
                    optimizer.step()

                    epoch_loss += loss.item()
                    batch_count += 1

                del X_sub, y_sub, train_ds, train_loader
                gc.collect()

            print(f"  Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / max(batch_count, 1):.4f}")

        model.eval()
        X_test = X_test[:, np.newaxis, :, :].astype(np.float32)
        test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test.astype(np.int64)))
        test_loader = DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )

        all_preds = []
        all_true = []

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device, non_blocking=True)
                outputs = model(xb)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                all_preds.extend(preds)
                all_true.extend(yb.numpy())

        all_preds = np.array(all_preds)
        all_true = np.array(all_true)

        acc = accuracy_score(all_true, all_preds)
        prec = precision_score(all_true, all_preds, zero_division=0)
        rec = recall_score(all_true, all_preds, zero_division=0)
        f1 = f1_score(all_true, all_preds, zero_division=0)
        f1_macro = f1_score(all_true, all_preds, average='macro', zero_division=0)
        cm = confusion_matrix(all_true, all_preds)

        print(f"  Fold Accuracy: {acc:.4f}")
        print(f"  Fold Precision: {prec:.4f}")
        print(f"  Fold Recall: {rec:.4f}")
        print(f"  Fold F1-score: {f1:.4f}")
        print(f"  Fold Macro F1: {f1_macro:.4f}")
        print(f"  Confusion Matrix:\n{cm}")

        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
        macro_f1_scores.append(f1_macro)
        conf_matrices.append(cm)

        del X_test, y_test, test_ds, test_loader, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std": np.std(accuracies),
        "precision_mean": np.mean(precisions),
        "precision_std": np.std(precisions),
        "recall_mean": np.mean(recalls),
        "recall_std": np.std(recalls),
        "f1_mean": np.mean(f1_scores),
        "f1_std": np.std(f1_scores),
        "macro_f1_mean": np.mean(macro_f1_scores),
        "macro_f1_std": np.std(macro_f1_scores),
        "confusion_matrices": conf_matrices
    }

In [77]:
deepconv_results_deap = run_loso_deepconvnet_from_disk(
    save_dir=save_dir_raw,
    num_subjects=32,
    num_channels=32,
    num_samples=1024,
    epochs=3,
    batch_size=64,
    lr=0.001,
)

Using device: cuda

Fold 1/32
Test class counts: [986 174]
Train class counts: [28333  7627]
  Epoch 1/3, Loss: 0.7830
  Epoch 2/3, Loss: 0.7533
  Epoch 3/3, Loss: 0.7408
  Fold Accuracy: 0.7767
  Fold Precision: 0.1600
  Fold Recall: 0.1149
  Fold F1-score: 0.1338
  Fold Macro F1: 0.5028
  Confusion Matrix:
[[881 105]
 [154  20]]

Fold 2/32
Test class counts: [986 174]
Train class counts: [28333  7627]
  Epoch 1/3, Loss: 0.7725
  Epoch 2/3, Loss: 0.7659
  Epoch 3/3, Loss: 0.7452
  Fold Accuracy: 0.5922
  Fold Precision: 0.1253
  Fold Recall: 0.2874
  Fold F1-score: 0.1745
  Fold Macro F1: 0.4519
  Confusion Matrix:
[[637 349]
 [124  50]]

Fold 3/32
Test class counts: [551 609]
Train class counts: [28768  7192]
  Epoch 1/3, Loss: 0.7756
  Epoch 2/3, Loss: 0.7412
  Epoch 3/3, Loss: 0.7349
  Fold Accuracy: 0.5147
  Fold Precision: 0.5215
  Fold Recall: 0.9146
  Fold F1-score: 0.6643
  Fold Macro F1: 0.3943
  Confusion Matrix:
[[ 40 511]
 [ 52 557]]

Fold 4/32
Test class counts: [1044  11

NameError: name 'deepconv_results' is not defined

In [1]:
print("\n===== DEAP LOSO DeepConvNet RESULTS =====")
print(f"Accuracy:  {deepconv_results_deap['accuracy_mean']:.4f} ± {deepconv_results_deap['accuracy_std']:.4f}")
print(f"Precision: {deepconv_results_deap['precision_mean']:.4f} ± {deepconv_results_deap['precision_std']:.4f}")
print(f"Recall:    {deepconv_results_deap['recall_mean']:.4f} ± {deepconv_results_deap['recall_std']:.4f}")
print(f"F1-score:  {deepconv_results_deap['f1_mean']:.4f} ± {deepconv_results_deap['f1_std']:.4f}")
print(f"Macro F1:  {deepconv_results_deap['macro_f1_mean']:.4f} ± {deepconv_results_deap['macro_f1_std']:.4f}")


===== DEAP LOSO DeepConvNet RESULTS =====


NameError: name 'deepconv_results_deap' is not defined